## Reconnaissance faciale avec InsightFace — version corrigée et nettoyée

Ce notebook reprend la logique de `face-recognition-insightface.ipynb`, avec les corrections suivantes :


In [ ]:
import os
import pickle
import json
from glob import glob
from datetime import datetime

import cv2
import numpy as np
from insightface.app.common import Face
from insightface.model_zoo import model_zoo

### Configuration

In [ ]:
EMBEDDINGS_PATH = "known_embeddings.npy"
NAMES_PATH = "known_names.pkl"
FICHIER_LOG = "acces.json"
DOSSIER_INCONNUS = "inconnus"
DOSSIER_SUCCES = "succes"
SEUIL_DEFAUT = 0.5

os.makedirs(DOSSIER_INCONNUS, exist_ok=True)
os.makedirs(DOSSIER_SUCCES, exist_ok=True)

### Chargement des modèles

In [ ]:
det_model = model_zoo.get_model("buffalo_l/det_10g.onnx", download=True)
rec_model = model_zoo.get_model("buffalo_l/w600k_r50.onnx", download=True)

det_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)
rec_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

### Extraction des embeddings connus

Parcourt `Dataset/<Personne>/*.jpg`, extrait un embedding par image et sauvegarde
dans `known_embeddings.npy` / `known_names.pkl`.

In [ ]:
def extract_face_embeddings(dataset_dir="Dataset"):
    if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(NAMES_PATH):
        known_embeddings = np.load(EMBEDDINGS_PATH)
        with open(NAMES_PATH, "rb") as f:
            known_names = pickle.load(f)
    else:
        known_embeddings = np.array([]).reshape(0, 512)
        known_names = []

    person_dirs = [
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ]
    print("Extraction des embeddings...")

    for person_name in person_dirs:
        directory = os.path.join(dataset_dir, person_name)
        img_paths = glob(f"{directory}/*.jpg")
        new_embeddings = []

        for img_path in img_paths:
            img = cv2.imread(img_path)
            if img is None:
                print(f"  Impossible de lire {img_path}")
                continue

            bboxes, kpss = det_model.detect(img, max_num=0, metric="default")
            if len(bboxes) == 0:
                print(f"  Aucun visage détecté dans {img_path}")
                continue

            # On suppose que le premier visage détecté est le bon
            bbox = bboxes[0, :4]
            det_score = bboxes[0, 4]
            kps = kpss[0]
            face = Face(bbox=bbox, kps=kps, det_score=det_score)
            rec_model.get(img, face)

            if hasattr(face, "normed_embedding"):
                new_embeddings.append(face.normed_embedding)
            else:
                print(f"  Échec d'extraction de l'embedding pour {img_path}")

        if new_embeddings:
            new_embeddings = np.vstack(new_embeddings)
            known_embeddings = np.vstack([known_embeddings, new_embeddings])
            known_names.extend([person_name] * new_embeddings.shape[0])

    np.save(EMBEDDINGS_PATH, known_embeddings)
    with open(NAMES_PATH, "wb") as f:
        pickle.dump(known_names, f)

    print("Tous les embeddings ont été extraits et sauvegardés.")
    return known_embeddings, known_names

In [ ]:
# Décommenter pour (re)générer la base à partir du dossier Dataset/
# extract_face_embeddings()

### Chargement des embeddings et recherche de correspondance

In [ ]:
def load_embeddings():
    """Recharge known_embeddings.npy / known_names.pkl. Retourne (None, None) si absents."""
    if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(NAMES_PATH):
        known_embeddings = np.load(EMBEDDINGS_PATH)
        with open(NAMES_PATH, "rb") as f:
            known_names = pickle.load(f)
        return known_embeddings, known_names

    print("Aucun embedding sauvegardé trouvé.")
    return None, None

In [ ]:
def find_match(embedding, known_embeddings, known_names, threshold=SEUIL_DEFAUT):
    """Similarité cosinus entre un embedding test et la base connue."""
    scores = np.dot(embedding, known_embeddings.T)
    scores = np.clip(scores, 0.0, 1.0)
    idx = np.argmax(scores)
    score = scores[idx]
    name = known_names[idx] if score > threshold else "Inconnu"
    return name, score

### Journal des accès (`acces.json`)

In [ ]:
def charger_logs():
    """Charge l'historique depuis acces.json. Retourne [] si absent ou corrompu."""
    if not os.path.exists(FICHIER_LOG):
        return []
    with open(FICHIER_LOG, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

In [ ]:
def enregistrer_acces(nom, statut, score, image=None):
    """
    Ajoute une entrée dans le journal.
    Paramètres : nom (str), statut (\'SUCCES\'|\'REFUSE\'), score (float 0-1),
                 image (str|None) — nom du fichier image sauvegardé.
    """
    logs = charger_logs()
    logs.append({
        "date": datetime.now().strftime("%Y-%m-%d"),
        "heure": datetime.now().strftime("%H:%M:%S"),
        "nom": nom,
        "statut": statut,
        "score_detection": round(float(score), 4),
        "image": image,
    })
    with open(FICHIER_LOG, "w", encoding="utf-8") as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)

### Recadrage visage + épaules (pour la sauvegarde image uniquement)

In [ ]:
def agrandir_bbox_epaules(bbox, frame_shape, marge_haut=0.5, marge_bas=0.8, marge_cotes=0.6):
    """
    Agrandit la bounding box du visage pour inclure les épaules.
    Utilisée uniquement pour la sauvegarde de l'image, jamais pour l'embedding.
    """
    x1, y1, x2, y2 = bbox.astype(int)
    h_frame, w_frame = frame_shape[:2]

    largeur = x2 - x1
    hauteur = y2 - y1

    x1_e = max(0, x1 - int(largeur * marge_cotes))
    y1_e = max(0, y1 - int(hauteur * marge_haut))
    x2_e = min(w_frame, x2 + int(largeur * marge_cotes))
    y2_e = min(h_frame, y2 + int(hauteur * marge_bas))

    return x1_e, y1_e, x2_e, y2_e

### Reconnaissance en temps réel (boucle webcam)

**Correction apportée** : le cas "Inconnu" utilisait `y1` (non défini à ce stade de la boucle)
au lieu de `y1_e` pour recadrer l'image sauvegardée. Les deux branches (connu/inconnu)
utilisent maintenant `x1_e / y1_e / x2_e / y2_e`. Le score loggé pour les succès est aussi
corrigé (`match_score` au lieu de `threshold`).

Fermer la fenêtre avec la touche **q**.

In [ ]:
def face_recognition_loop(known_embeddings, known_names, threshold=SEUIL_DEFAUT):
    cap = cv2.VideoCapture(0, cv2.CAP_MSMF)
    if not cap.isOpened():
        print("Impossible d'ouvrir la caméra.")
        return

    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print("Erreur : frame non capturée")
            break

        bboxes, kpss = det_model.detect(frame, max_num=0, metric="default")

        for i in range(len(bboxes)):
            bbox = bboxes[i, :4]
            kps = kpss[i]
            face = Face(bbox=bbox, kps=kps, det_score=bboxes[i, 4])
            rec_model.get(frame, face)
            test_embedding = face.normed_embedding

            pred_name, match_score = find_match(
                test_embedding, known_embeddings, known_names, threshold
            )

            x1_e, y1_e, x2_e, y2_e = agrandir_bbox_epaules(bbox, frame.shape)
            visage = frame[y1_e:y2_e, x1_e:x2_e]

            horodatage = datetime.now().strftime("%Y%m%d_%H%M%S")

            if pred_name == "Inconnu":
                color = (0, 0, 255)
                label = pred_name
                nom_fichier = f"inconnu_{horodatage}.jpg"
                cv2.imwrite(os.path.join(DOSSIER_INCONNUS, nom_fichier), visage)
                enregistrer_acces("Inconnu", "REFUSE", match_score, image=nom_fichier)
            else:
                color = (0, 255, 0)
                label = f"{pred_name} ({match_score:.2f})"
                nom_fichier = f"{pred_name}_{horodatage}.jpg"
                cv2.imwrite(os.path.join(DOSSIER_SUCCES, nom_fichier), visage)
                enregistrer_acces(pred_name, "SUCCES", match_score, image=nom_fichier)

            x1, y1, x2, y2 = map(int, bbox)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 2)

        cv2.imshow("Reconnaissance faciale", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

### Lancement

In [ ]:
known_embeddings, known_names = load_embeddings()
if known_embeddings is not None and known_names is not None:
    face_recognition_loop(known_embeddings, known_names)
else:
    print("Lance extract_face_embeddings() d'abord pour générer la base.")